# 02. Deep Agent의 하네스(Harness) 구성 능력

> 🔑 **비유**: Deep Agent의 구성 능력은 **등산 장비 세트**와 같아요. 로프(파일시스템), 나침반(계획), 무전기(서브에이전트), 산소통(컨텍스트 관리), 다목적 칼(코드 실행), 비상 호루라기(HITL) — 이 장비들을 갖추면 어떤 산이든 오를 수 있어요.
>
> 공식 [Harness 문서](https://docs.langchain.com/oss/python/deepagents/harness)는 **7가지 능력**(Planning · Virtual Filesystem · Filesystem Permissions · Subagents · Context · Code Execution · HITL)을 열거해요. 이 노트북에서는 그 중 Permissions를 제외한 **6가지를 직접 코드로 실습**하고, Permissions는 Filesystem 섹션에서 선언형 API만 간단히 소개해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. Deep Agent의 핵심 능력(Planning, Filesystem, Subagents, Context, Code Execution, HITL)을 직접 코드로 실습할 수 있어요
2. `backend` 파라미터로 다양한 백엔드(`StateBackend`, `FilesystemBackend`, `CompositeBackend`)를 설정할 수 있어요
3. `FilesystemPermission` 으로 경로·연산별 허용/거부 규칙을 선언할 수 있어요
4. `interrupt_on` 파라미터와 `Command`로 인간-루프(HITL)를 구현할 수 있어요
5. 커스텀 `BackendProtocol`을 구현해서 나만의 백엔드를 만들 수 있어요

## 사전 지식

- 이전 노트북 `01-Deep-Agents-Overview.ipynb`: `create_deep_agent` 기본 사용법
- LangGraph 체크포인터(`InMemorySaver`) 개념
- LangGraph `interrupt` 및 `Command` (Part 2에서 학습)

## 이 노트북에서 실습하는 6가지 능력 + Permissions 개요

공식 [Harness capabilities](https://docs.langchain.com/oss/python/deepagents/harness) 는 Planning, Virtual Filesystem, **Filesystem Permissions**, Subagents, Context Management, Code Execution, HITL 등 **7가지**를 열거해요. 이 노트북에서는 그 중 여섯 능력을 직접 코드로 실습하고, 나머지 **Permissions**는 Filesystem 섹션에서 `FilesystemPermission` 선언 예시만 짧게 소개해요. 각 능력은 파라미터·도구·미들웨어 중 하나 이상으로 제어돼요.

```mermaid
flowchart TD
    CDA["create_deep_agent<br>(진입점)"]

    subgraph PLAN["1. Planning"]
        P1["write_todos 도구<br>(자동 내장)"] --> P2["할일 목록 생성<br>단계별 작업 분해"]
    end

    subgraph FS["2. Filesystem (+Permissions)"]
        F1["backend 파라미터"] --> F2["ls / read_file / write_file / edit_file / glob / grep"]
        F3["permissions= 파라미터"] --> F4["FilesystemPermission<br>경로·연산별 allow/deny"]
    end

    subgraph SUB["3. Subagents"]
        S1["subagents 파라미터"] --> S2["task 도구<br>하위 에이전트 위임"]
    end

    subgraph CTX["4. Context Management"]
        C1["오프로딩<br>(큰 도구 결과를 파일로)"] --> C2["요약 미들웨어<br>(히스토리가 커지면 동작)"]
    end

    subgraph CODE["5. Code Execution"]
        E1["execute 도구<br>(샌드박스 연동)"] --> E2["LocalShellBackend /<br>Modal · Daytona · Runloop 등"]
    end

    subgraph HITL["6. HITL"]
        H1["interrupt_on 파라미터"] --> H2["checkpointer 필수<br>Command로 재개"]
    end

    CDA --> PLAN
    CDA --> FS
    CDA --> SUB
    CDA --> CTX
    CDA --> CODE
    CDA --> HITL

    classDef entry fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef ability fill:#cce5ff,stroke:#007bff,color:#004085
    classDef detail fill:#d4edda,stroke:#28a745,color:#155724

    class CDA entry
    class PLAN,FS,SUB,CTX,CODE,HITL ability
    class P1,P2,F1,F2,F3,F4,S1,S2,C1,C2,E1,E2,H1,H2 detail
```

| # | 능력 | 설정 방법 | 핵심 API |
|---|------|-----------|----------|
| 1 | **계획(Planning)** | 자동 내장 | `write_todos` 도구 |
| 2 | **파일시스템(Filesystem)** | `backend=` 파라미터 | `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep` |
| 2-A | **권한(Permissions)** | `permissions=` 파라미터 | `FilesystemPermission(operations=..., paths=..., mode="deny"|"allow")` |
| 3 | **서브에이전트(Subagents)** | `subagents=` 파라미터 | `task` 도구 |
| 4 | **컨텍스트 관리** | 오프로딩 + 요약 미들웨어 | 큰 도구 입출력은 파일로 이관, 히스토리는 요약 |
| 5 | **코드 실행** | `backend=` 파라미터 (샌드박스) | `execute` 도구 |
| 6 | **HITL** | `interrupt_on=` 파라미터 | `Command(resume=...)` |


## 환경 설정

In [1]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# .env 파일에서 OPENAI_API_KEY 등을 로드해요
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택 사항)
# ---------------------------------------------------
import os
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-Deep-Agents-Harness"


## 1. Planning (계획) — write_todos

Deep Agents가 일반 에이전트와 가장 다른 점 중 하나는 **계획 능력**이에요. `write_todos` 도구가 자동으로 내장되어 있어서, 복잡한 작업을 받으면 에이전트가 스스로 단계별 할일 목록을 작성하고 하나씩 수행해요.

```mermaid
flowchart LR
    INPUT["복잡한 작업 요청"] --> PLAN["write_todos<br>할일 목록 작성"]
    PLAN --> T1["할일 1: 자료 수집"]
    PLAN --> T2["할일 2: 분석 수행"]
    PLAN --> T3["할일 3: 결과 작성"]
    T1 --> EXEC["순서대로 실행"]
    T2 --> EXEC
    T3 --> EXEC
    EXEC --> OUTPUT["최종 결과 반환"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef plan fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef task fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class INPUT input
    class PLAN plan
    class T1,T2,T3 task
    class EXEC,OUTPUT output
```

> 🔑 **핵심 개념**: `write_todos`는 `create_deep_agent`가 에이전트의 도구 목록에 **자동으로 추가**해요. 별도 설정 없이도 에이전트가 복잡한 작업을 받으면 먼저 계획부터 세우도록 동작해요. 이것이 단순 ReAct 에이전트(`create_agent`)와의 가장 큰 차이점이에요.

> 🎯 **강의 포인트**: 학생들에게 "에이전트에게 단순 질문이 아니라 '조사 보고서를 써줘' 같은 복잡한 작업을 주면 `write_todos`가 자동으로 실행되는 걸 볼 수 있어요"라고 강조해주세요. 직접 복잡한 과제를 주고 출력을 관찰하면 효과적이에요.

In [3]:
# ---------------------------------------------------
# Planning 능력 시연: write_todos 자동 동작 확인
# ---------------------------------------------------
# Deep Agents의 핵심 진입점과 메시지 타입을 가져와요
from deepagents import create_deep_agent
from langchain.messages import HumanMessage

# ---------------------------------------------------
# Planning 전용 에이전트 생성
# ---------------------------------------------------
# tools가 비어 있어도 write_todos는 자동으로 내장돼요
planning_agent = create_deep_agent(
    model="openai:gpt-4o-mini",  # 기본 모델
    tools=[],                     # 추가 도구 없음
    system_prompt=(
        "당신은 프로젝트 계획을 세우는 전문가입니다. "
        "복잡한 작업을 받으면 반드시 단계별 할일 목록을 먼저 작성한 뒤 수행하세요."
    ),
)

# ---------------------------------------------------
# 복잡한 작업 요청 → write_todos 자동 실행 유도
# ---------------------------------------------------
# 단순 질문이 아닌 "여러 단계가 필요한 작업"을 주면
# 에이전트가 스스로 계획을 세워요
complex_task = (
    "LangGraph와 Deep Agents에 대한 간단한 학습 계획서를 작성해줘. "
    "목표 설정 → 학습 단계 → 실습 과제 → 평가 방법 순서로 구성해줘."
)

# 복잡한 작업 요청 실행 중...
# --------------------------------------------------

# stream_mode="updates"로 각 노드의 실행 과정을 볼 수 있어요
# Deep Agents 노드는 None이나 Overwrite 객체를 반환하는 경우가 있으므로 방어 처리해요
for chunk in planning_agent.stream(
    {"messages": [HumanMessage(content=complex_task)]},
    stream_mode="updates"
):
    for node_name, node_output in chunk.items():
        if not node_output or not isinstance(node_output, dict):
            continue
        if "messages" in node_output:
            msgs = node_output["messages"]
            if hasattr(msgs, "value"):
                msgs = msgs.value
            if not isinstance(msgs, list):
                continue
            for msg in msgs:
                # write_todos 도구 호출이 포함된 메시지인지 확인해요
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"[도구 호출] {tc['name']}")
                        if "todos" in str(tc.get("args", "")):
                            #   → 할일 목록 자동 작성 중...
                            pass
                elif hasattr(msg, "content") and msg.content:
                    content = msg.content
                    # content가 list(Content Blocks)인 경우 텍스트 추출
                    if isinstance(content, list):
                        content = " ".join(
                            b.get("text", "") if isinstance(b, dict) else str(b)
                            for b in content
                        )
                    print(f"[{node_name}]:\n{str(content)[:500]}")


[PatchToolCallsMiddleware.before_agent]:
LangGraph와 Deep Agents에 대한 간단한 학습 계획서를 작성해줘. 목표 설정 → 학습 단계 → 실습 과제 → 평가 방법 순서로 구성해줘.
[도구 호출] write_todos
[tools]:
Updated todo list to [{'content': '목표 설정: LangGraph 및 Deep Agents에 대한 기본 이해와 활용 가능 목표 설정', 'status': 'in_progress'}, {'content': '학습 단계: LangGraph의 정의, 특징, 활용 예제 및 Deep Agents의 기능과 적용 사례 조사', 'status': 'pending'}, {'content': '실습 과제: LangGraph와 Deep Agents를 사용하여 간단한 프로젝트 구현하기', 'status': 'pending'}, {'content': '평가 방법: 프로젝트 결과물 및 구현 과정에 대한 피드백 및 자가 평가 기준 마련하기', 'status': 'pending'}]
[도구 호출] write_todos
[tools]:
Updated todo list to [{'content': '목표 설정: LangGraph 및 Deep Agents에 대한 기본 이해와 활용 가능 목표 설정', 'status': 'completed'}, {'content': '학습 단계: LangGraph의 정의, 특징, 활용 예제 및 Deep Agents의 기능과 적용 사례 조사', 'status': 'in_progress'}, {'content': '실습 과제: LangGraph와 Deep Agents를 사용하여 간단한 프로젝트 구현하기', 'status': 'pending'}, {'content': '평가 방법: 프로젝트 결과물 및 구현 과정에 대한 피드백 및 자가 평가 기준 마련하기', 'status': 'pending'}]
[도구 호출] write_todos
[tools]:
Updated todo 

## 2. Filesystem (파일시스템) — Backend 설정

Deep Agents의 파일시스템 능력은 `backend` 파라미터로 제어해요. 백엔드 종류에 따라 에이전트가 접근할 수 있는 파일 범위가 달라져요.

### 백엔드 종류

| 백엔드 | 특징 | 사용 시나리오 |
|--------|------|---------------|
| `StateBackend` | 인메모리, 일시적 저장 | 테스트, 프로토타이핑 |
| `FilesystemBackend` | 로컬 파일시스템 접근 | 로컬 개발, 파일 조작 |
| `StoreBackend` | 스레드 간 공유 저장소 | 장기 메모리, 크로스 세션 |
| `LocalShellBackend` | 셸 명령 + 파일시스템 | 로컬 개발 환경 |
| `CompositeBackend` | 경로별 다른 백엔드 라우팅 | 복합 환경 |
| Custom | `BackendProtocol` 구현 | 특수 요구사항 |

> 💡 **실무 팁**: 프로덕션 환경에서는 `CompositeBackend`를 자주 사용해요. 예를 들어, `/tmp/` 경로는 `LocalShellBackend`, `/store/` 경로는 `StoreBackend`로 라우팅하면 파일 용도에 따라 다른 저장소를 쓸 수 있어요.

> ⚠️ **자주 하는 실수**: `FilesystemBackend`를 사용할 때 `root_dir`을 지정하지 않으면 에이전트가 시스템 전체 파일에 접근할 수 있어요. 항상 작업 디렉토리를 명시적으로 제한하는 것이 안전해요.

In [4]:
# ---------------------------------------------------
# 백엔드 종류 소개 및 StateBackend (인메모리) 시연
# ---------------------------------------------------
# Deep Agents의 백엔드 모듈에서 필요한 클래스를 가져와요
from deepagents.backends import StateBackend

# ---------------------------------------------------
# 각 백엔드의 인터페이스 설명 (설치 없이도 개념 이해)
# ---------------------------------------------------
# Deep Agents 백엔드 종류와 용도:
# ==================================================

backends = [
    ("StateBackend",     "인메모리, 세션 내 임시 저장",         "테스트, 프로토타입"),
    ("FilesystemBackend","로컬 디스크 접근 (root_dir 제한)",    "로컬 파일 조작"),
    ("StoreBackend",     "LangGraph Store API 연동",           "크로스 세션 장기 메모리"),
    ("LocalShellBackend","셸 실행 + 파일시스템 통합",           "로컬 개발 환경 전체"),
    ("CompositeBackend", "경로별 백엔드 라우팅",               "복합 저장소 환경"),
]

for name, desc, use_case in backends:
    print(f"  {name:22s} | {desc:35s} | 용도: {use_case}")


  StateBackend           | 인메모리, 세션 내 임시 저장                    | 용도: 테스트, 프로토타입
  FilesystemBackend      | 로컬 디스크 접근 (root_dir 제한)             | 용도: 로컬 파일 조작
  StoreBackend           | LangGraph Store API 연동              | 용도: 크로스 세션 장기 메모리
  LocalShellBackend      | 셸 실행 + 파일시스템 통합                     | 용도: 로컬 개발 환경 전체
  CompositeBackend       | 경로별 백엔드 라우팅                         | 용도: 복합 저장소 환경


In [ ]:
# ---------------------------------------------------
# FilesystemBackend로 파일 읽기/쓰기 에이전트 만들기
# ---------------------------------------------------
import os
import tempfile
from pathlib import Path

# 실습용 임시 디렉토리를 생성해요
# 실제 프로젝트에서는 작업 디렉토리를 명시적으로 지정해요
work_dir = tempfile.mkdtemp(prefix="deep_agents_demo_")
print(f"작업 디렉토리: {work_dir}")

# 실습용 샘플 파일을 미리 만들어둬요
sample_file = Path(work_dir) / "notes.txt"
sample_file.write_text(
    "LangGraph 학습 노트\n"
    "===================\n"
    "1. StateGraph: 상태 기반 그래프\n"
    "2. Node: 그래프의 실행 단위\n"
    "3. Edge: 노드 간 전환 규칙\n"
    "4. Checkpointer: 상태 저장소\n",
    encoding="utf-8"
)
print(f"샘플 파일 생성: {sample_file.name}")

# ---------------------------------------------------
# FilesystemBackend 생성 및 에이전트 연결
# ---------------------------------------------------
# root_dir로 접근 범위를 제한해요 (보안 중요)
from deepagents.backends import FilesystemBackend

fs_agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    tools=[],  # 파일 도구는 backend에서 자동 주입돼요
    system_prompt=(
        "당신은 파일 관리 에이전트입니다. "
        "사용자가 요청한 파일 작업을 정확하게 수행하세요."
    ),
    backend=FilesystemBackend(
        root_dir=work_dir,
        virtual_mode=True,  # 현재 버전에서는 명시 권장: 경고 방지 + 가상 경로 semantics
    ),
)

# 에이전트에게 파일 읽기 요청
result = fs_agent.invoke({
    "messages": [HumanMessage(
        content="notes.txt 파일의 내용을 읽어서 요약해줘"
    )]
})

# 에이전트 응답:
print(result["messages"][-1].content)


### CompositeBackend 설정 예시

**공식 API 시그니처**: `CompositeBackend(default=<Backend>, routes={"/prefix/": <Backend>, ...})` — `routes`는 **딕셔너리**로, 키는 경로 프리픽스, 값은 해당 경로를 처리할 백엔드예요 ([Backends 공식 문서](https://docs.langchain.com/oss/python/deepagents/backends)).

```python
from deepagents import create_deep_agent
from deepagents.backends import (
    CompositeBackend,
    FilesystemBackend,
    StateBackend,
    StoreBackend,
)
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

# 경로별로 다른 백엔드를 지정해요 (routes는 dict)
composite = CompositeBackend(
    default=StateBackend(),              # 매칭 없을 때 기본 백엔드
    routes={
        "/memories/": StoreBackend(      # 크로스 스레드 영구 저장
            namespace=lambda rt: (rt.server_info.assistant_id,),
        ),
        "/workspace/": FilesystemBackend(  # 로컬 디스크
            root_dir="./workspace",
            virtual_mode=True,
        ),
    },
)

agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    backend=composite,
)
# /memories/agent.md -> StoreBackend (영구 저장)
# /workspace/plan.md -> FilesystemBackend (로컬 디스크)
# /tmp/scratch.txt  -> StateBackend (임시, 세션 내)
```

> 💡 **실무 팁**: `StoreBackend(namespace=...)` 콜백은 `ToolRuntime`을 받아 어떤 식별자로 데이터를 격리할지 결정해요. 에이전트 전역 메모리는 `assistant_id`, 사용자별 메모리는 `user_id`를 사용하세요.

## 3. Subagents (서브에이전트) — 작업 위임

복잡한 작업을 여러 전문 에이전트에게 나눠서 처리할 수 있어요. 메인 에이전트가 `task` 도구로 서브에이전트에게 하위 작업을 위임하는 방식이에요.

```mermaid
flowchart TD
    USER["사용자 요청"] --> MAIN["메인 에이전트<br>create_deep_agent"]
    MAIN -->|"task 도구 호출"| S1["서브에이전트 1<br>자료 조사 전문"]
    MAIN -->|"task 도구 호출"| S2["서브에이전트 2<br>코드 작성 전문"]
    MAIN -->|"task 도구 호출"| S3["서브에이전트 3<br>문서 작성 전문"]
    S1 -->|"결과 반환"| MAIN
    S2 -->|"결과 반환"| MAIN
    S3 -->|"결과 반환"| MAIN
    MAIN --> OUTPUT["통합 결과 반환"]

    classDef user fill:#d4edda,stroke:#28a745,color:#155724
    classDef main fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef sub fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class USER user
    class MAIN main
    class S1,S2,S3 sub
    class OUTPUT output
```

### 서브에이전트 정의 방식

| 방식 | 코드 | 특징 |
|------|------|------|
| 딕셔너리(dict) | `{"name": "...", "description": "..."}` | 간단, 빠른 정의 |
| `CompiledSubAgent` | `CompiledSubAgent(graph=my_graph, ...)` | 기존 LangGraph 그래프 재활용 |
| 기본 제공 | `subagents=None` (기본값) | general-purpose 에이전트 자동 포함 |

> 🔑 **핵심 개념**: `subagents` 파라미터를 생략하면(기본값 `None`) Deep Agents가 **general-purpose 서브에이전트**를 자동으로 포함시켜요. 명시적으로 `[]`를 전달하면 서브에이전트 없이 동작해요.

> 💡 **실무 팁**: 특정 도메인 전문성이 필요한 서브에이전트는 딕셔너리로 정의할 때 `description`을 아주 구체적으로 작성해야 해요. 메인 에이전트가 이 설명을 읽고 어떤 작업을 어떤 서브에이전트에게 넘길지 결정하기 때문이에요.

In [6]:
# ---------------------------------------------------
# 서브에이전트 딕셔너리 방식 정의
# ---------------------------------------------------
# 서브에이전트를 딕셔너리로 간단하게 정의할 수 있어요
# name: 서브에이전트 식별자
# description: 메인 에이전트가 참고하는 역할 설명
# model: 서브에이전트에 사용할 모델
# system_prompt: 서브에이전트의 역할 지침 (필수)

researcher_subagent = {
    "name": "researcher",
    "description": (
        "인터넷 검색과 자료 수집 전문가입니다. "
        "특정 주제에 대한 정보를 수집하고 요약하는 작업을 담당해요."
    ),
    "model": "openai:gpt-4o-mini",
    "system_prompt": (
        "당신은 자료 조사 전문가입니다. "
        "주어진 주제에 대해 핵심 정보를 수집하고 간결하게 요약해주세요."
    ),
    "tools": [],
}

writer_subagent = {
    "name": "writer",
    "description": (
        "문서 작성 전문가입니다. "
        "수집된 정보를 바탕으로 구조적인 보고서나 문서를 작성하는 작업을 담당해요."
    ),
    "model": "openai:gpt-4o-mini",
    "system_prompt": (
        "당신은 전문 작가입니다. "
        "수집된 정보를 바탕으로 명확하고 구조적인 문서를 작성해주세요."
    ),
    "tools": [],
}

# 서브에이전트 정의 완료:
for sa in [researcher_subagent, writer_subagent]:
    print(f"  - {sa['name']}: {sa['description'][:60]}...")


  - researcher: 인터넷 검색과 자료 수집 전문가입니다. 특정 주제에 대한 정보를 수집하고 요약하는 작업을 담당해요....
  - writer: 문서 작성 전문가입니다. 수집된 정보를 바탕으로 구조적인 보고서나 문서를 작성하는 작업을 담당해요....


### CompiledSubAgent 사용 예시

```python
from deepagents import create_deep_agent, CompiledSubAgent
from langgraph.graph import StateGraph

# 기존 LangGraph 그래프를 먼저 만들어요 (Part 2, 4에서 배운 방식)
existing_graph = StateGraph(...)  # 이미 만들어둔 그래프
compiled = existing_graph.compile()

# CompiledSubAgent로 래핑해서 서브에이전트로 사용해요
custom_subagent = CompiledSubAgent(
    name="custom_analyzer",
    description="특수 데이터 분석 전문 에이전트입니다.",
    graph=compiled,  # 기존 그래프 재활용!
)

# 메인 에이전트에 서브에이전트로 등록
main_agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    subagents=[custom_subagent, researcher_subagent],
)
```

> 💡 **실무 팁**: CompiledSubAgent는 기존 LangGraph 자산을 재활용할 수 있어요. 특수 로직(RAG, 커스텀 도구 등)이 있는 그래프를 서브에이전트로 사용해요.


In [7]:
# ---------------------------------------------------
# 서브에이전트를 포함한 메인 에이전트 생성
# ---------------------------------------------------
# 서브에이전트 목록을 메인 에이전트에 등록해요
main_with_subagents = create_deep_agent(
    model="openai:gpt-4o-mini",
    tools=[],
    system_prompt=(
        "당신은 복잡한 작업을 조율하는 오케스트레이터 에이전트입니다. "
        "자료 조사는 researcher 서브에이전트에게, "
        "문서 작성은 writer 서브에이전트에게 위임하세요."
    ),
    # subagents: 딕셔너리 또는 CompiledSubAgent 리스트
    subagents=[researcher_subagent, writer_subagent],
)

# 서브에이전트 등록 완료:
print(f"  - researcher (조사 담당)")
print(f"  - writer (문서 작성 담당)")
print()
# 에이전트 준비 완료. 복잡한 작업을 위임할 수 있어요.


  - researcher (조사 담당)
  - writer (문서 작성 담당)



## 4. Context Management (컨텍스트 관리) — 자동 압축

Deep Agents는 대화가 길어질 때 컨텍스트를 자동으로 관리해요. 토큰 수가 임계값(20K+)을 초과하면 오래된 메시지를 요약하여 전체 컨텍스트 길이를 줄여요.

```mermaid
flowchart LR
    MSG["대화 히스토리<br>증가 중"] --> CHECK{"20K+ 토큰<br>초과?"}  
    CHECK -->|"아니오"| KEEP["그대로 유지"]
    CHECK -->|"예"| COMPRESS["자동 압축<br>요약 비율 85%"]
    COMPRESS --> SUMMARY["요약된 메시지\ + 최근 메시지"]
    SUMMARY --> CONTINUE["작업 계속"]
    KEEP --> CONTINUE

    classDef check fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef compress fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef normal fill:#d4edda,stroke:#28a745,color:#155724

    class CHECK check
    class COMPRESS,SUMMARY compress
    class MSG,KEEP,CONTINUE normal
```

### 자동 압축 동작 방식

| 항목 | 값 | 설명 |
|------|-----|------|
| 압축 임계값 | 20,000 토큰 | 이 값 초과 시 압축 시작 |
| 요약 비율 | 85% | 전체의 85%를 요약으로 대체 |
| 유지 메시지 | 최근 15% | 최근 메시지는 원본 유지 |
| 압축 방식 | LLM 요약 | 같은 모델로 내용 압축 |

> 🔑 **핵심 개념**: 이 자동 압축 덕분에 Deep Agents는 이론적으로 **무한 길이의 작업**을 처리할 수 있어요. 수십 분에 걸쳐 실행되는 작업도 컨텍스트 초과로 멈추지 않아요.

> ⚠️ **자주 하는 실수**: 압축이 일어나면 오래된 세부 정보가 요약본으로 대체돼요. 에이전트가 초반에 처리한 구체적인 값(파일 내용, 코드 등)을 나중에 정확히 기억하지 못할 수 있어요. 중요한 정보는 파일이나 Store에 저장하도록 에이전트에게 지시하는 것이 좋아요.

### 컨텍스트 관리 동작 흐름

**[1단계] 대화 시작 -> 메시지 누적**

| 메시지 | 토큰 |
|--------|------|
| HumanMessage(content='작업1...') | ~500 |
| AIMessage(content='결과1...') | ~1,200 |
| HumanMessage(content='작업2...') | ~800 |
| ... (중간 대화 20개) | ~18,000 |
| 최근 메시지들 | ~3,600 |
| **합계** | **~25,000 토큰** |

**[2단계] 20,000 토큰 초과 -> 자동 압축 트리거**
- 오래된 메시지 85% -> LLM이 요약 생성
- 최근 메시지 15% -> 원본 유지
- 결과: 약 6,600 토큰으로 압축 (74% 감소)

**[3단계] 압축 후 작업 계속**
- 에이전트는 압축된 히스토리를 기반으로 계속 동작

> 💡 **권장 사항**: 중요한 정보는 파일이나 Store에 저장하세요.
> - `write_todos`로 현재까지의 진행 상황 기록
> - 파일시스템 백엔드에 중간 결과 저장
> - Store API로 키-값 저장


## 5. Code Execution (코드 실행) — Sandbox as Tool

Deep Agents는 코드를 직접 실행하는 능력을 가질 수 있어요. 단, 보안을 위해 **샌드박스 환경**에서만 코드를 실행해요.

### 지원 샌드박스

| 샌드박스 | 설명 | 특징 |
|----------|------|------|
| **AgentCore** | AWS 기반 관리형 샌드박스 | 엔터프라이즈급, 완전 관리형 |
| **Modal** | 클라우드 함수 실행 | 빠른 시작, GPU 지원 |
| **Daytona** | 개발 환경 샌드박스 | 코드 실행 전문 |
| **Runloop** | 격리된 실행 환경 | 보안 강화 |

### Sandbox as Tool 패턴

공식 문서에서 권장하는 패턴이에요. 코드 실행 기능을 별도의 도구(Tool)로 감싸서 에이전트에게 제공해요.

> 🎯 **강의 포인트**: "Sandbox as Tool 패턴을 쓰면 코드 실행 백엔드를 나중에 쉽게 교체할 수 있어요. 개발 중에는 로컬 실행, 배포 시에는 Modal 같은 클라우드 샌드박스로 전환하면 돼요"라고 강조해주세요.

> ⚠️ **자주 하는 실수**: 샌드박스 없이 로컬 `exec()`나 `subprocess`로 코드를 실행하면 보안 위험이 있어요. Deep Agents의 `execute` 도구는 반드시 격리된 샌드박스를 사용해야 해요.

In [8]:
# ---------------------------------------------------
# Sandbox as Tool 패턴 구현
# ---------------------------------------------------
# 코드 실행을 안전하게 도구로 래핑하는 권장 패턴이에요

from langchain.tools import tool

@tool
def execute_python_safe(code: str) -> str:
    """안전한 샌드박스 환경에서 Python 코드를 실행해요.

    실제 프로덕션에서는 Modal, Daytona, Runloop 등
    격리된 샌드박스 API를 호출해야 해요.
    이 함수는 패턴 이해를 위한 데모 버전이에요.

    Args:
        code: 실행할 Python 코드 문자열

    Returns:
        코드 실행 결과 또는 에러 메시지
    """
    # ---------------------------------------------------
    # [데모] 안전한 제한 실행 (실제로는 샌드박스 API 호출)
    # ---------------------------------------------------
    import io
    import contextlib

    # 허용된 내장 함수만 사용 가능한 제한 환경이에요
    # 실제 프로덕션에서는 이 부분을 Modal/Daytona API로 교체해요
    safe_globals = {
        "__builtins__": {
            "print": print,
            "len": len,
            "range": range,
            "list": list,
            "dict": dict,
            "str": str,
            "int": int,
            "float": float,
            "sum": sum,
            "max": max,
            "min": min,
        }
    }

    # 표준 출력을 캡처해요
    stdout_capture = io.StringIO()

    try:
        with contextlib.redirect_stdout(stdout_capture):
            exec(code, safe_globals)  # noqa: S102
        output = stdout_capture.getvalue()
        return f"실행 성공:\n{output}" if output else "실행 성공 (출력 없음)"
    except Exception as e:
        return f"실행 오류: {type(e).__name__}: {e!s}"


# ---------------------------------------------------
# Sandbox as Tool 패턴으로 에이전트 생성
# ---------------------------------------------------
code_agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    tools=[execute_python_safe],  # 코드 실행 도구를 명시적으로 전달해요
    system_prompt=(
        "당신은 Python 코드를 작성하고 실행하는 전문가입니다. "
        "코드를 작성한 후 반드시 execute_python_safe 도구로 실행해서 결과를 확인하세요."
    ),
)

# 코드 실행 요청
result = code_agent.invoke({
    "messages": [HumanMessage(
        content="1부터 10까지의 합을 계산하는 Python 코드를 작성하고 실행해줘"
    )]
})

# 코드 실행 에이전트 응답:
print(result["messages"][-1].content)


[{'type': 'text', 'text': '1부터 10까지의 합은 55입니다.', 'annotations': [], 'id': 'msg_0c76a5f677b476ca006a2f7e40f310819b9a16bf8202ee9824'}]


### 실제 샌드박스 연동 예시 (참고용)

실제 프로덕션에서는 아래처럼 샌드박스 서비스 API를 호출해요.

**Modal 샌드박스**

```python
import modal

@tool
def execute_python_modal(code: str) -> str:
    # Modal 샌드박스에서 코드를 실행해요
    app = modal.App.lookup("code-executor")
    result = app.execute_code.remote(code)
    return result
```

**Daytona 샌드박스**

```python
from daytona_sdk import DaytonaClient

@tool
def execute_python_daytona(code: str) -> str:
    # Daytona 샌드박스에서 코드를 실행해요
    client = DaytonaClient()
    sandbox = client.get_sandbox("my-sandbox")
    result = sandbox.process.execute_command(f"python3 -c '{code}'")
    return result.output
```

> 💡 **실무 팁**: 'Sandbox as Tool' 패턴의 장점:
> 1. 코드 실행 백엔드를 언제든 교체 가능
> 2. 개발(로컬) -> 배포(클라우드) 전환이 코드 한 줄 변경
> 3. 에이전트 코드는 항상 동일, 백엔드만 달라짐


## 6. HITL (Human-In-The-Loop) — 인간 승인

Deep Agents의 HITL은 `interrupt_on` 파라미터로 설정해요. HITL의 기본 필요성과 `Command(resume=...)` 흐름은 `06_Middleware/02-Human-In-The-Loop-V1.ipynb`에서 이미 다뤘으므로, 여기서는 **Deep Agents에서 같은 승인 정책을 하네스 능력으로 켜는 방법**만 확인합니다.

```mermaid
flowchart TD
    START["Deep Agent 실행"] --> TOOL_CALL["도구 호출 시도"]
    TOOL_CALL --> CHECK{"interrupt_on 대상?"}
    CHECK -->|아니오| EXEC_DIRECT["바로 실행"]
    CHECK -->|예| INTERRUPT["승인 대기"]
    INTERRUPT --> HUMAN["approve / edit / reject"]
    HUMAN --> RESUME["Command(resume=...)"]
    RESUME --> NEXT["다음 단계"]
    EXEC_DIRECT --> NEXT

    classDef check fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef interrupt fill:#f8d7da,stroke:#dc3545,color:#721c24
    class CHECK check
    class INTERRUPT interrupt
```

### Deep Agents에서 확인할 차이

| 항목 | Middleware HITL | Deep Agents HITL |
|---|---|---|
| 승인 정책 | `HumanInTheLoopMiddleware` | `create_deep_agent(..., interrupt_on=...)` |
| 저장/재개 | checkpointer 필요 | checkpointer 필요 |
| 새 초점 | 도구 승인 정책 자체 | planning/filesystem/subagents와 함께 쓰는 하네스 능력 |

> ⚠️ **운영 주의**: Deep Agents에서도 interrupt 재개에는 checkpointer와 동일한 `thread_id`가 필요합니다. 이 규칙은 LangGraph 런타임 공통 규칙이에요.


In [9]:
# ---------------------------------------------------
# HITL 설정: interrupt_on + checkpointer
# ---------------------------------------------------
# InMemorySaver: 메모리 기반 체크포인터 (개발/테스트용)
from langgraph.checkpoint.memory import InMemorySaver
import uuid

# 파일 쓰기 도구 정의 (HITL 대상으로 사용할 도구예요)
from langchain.tools import tool

@tool
def write_file(filename: str, content: str) -> str:
    """파일을 생성하거나 수정해요. 중요한 작업이므로 인간 승인이 필요해요.

    Args:
        filename: 저장할 파일 이름
        content: 파일에 저장할 내용

    Returns:
        저장 성공 메시지
    """
    # 실제 파일 쓰기 (데모용이므로 출력만 해요)
    return f"'{filename}' 파일에 {len(content)}자 저장 완료"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """이메일을 발송해요. 인간 승인이 필요한 중요 작업이에요.

    Args:
        to: 수신자 이메일 주소
        subject: 이메일 제목
        body: 이메일 본문

    Returns:
        발송 성공 메시지
    """
    return f"'{to}'에게 '{subject}' 이메일 발송 완료"


# ---------------------------------------------------
# HITL 에이전트 생성
# ---------------------------------------------------
# interrupt_on: 딕셔너리로 도구별 인터럽트 설정
#   {"도구명": True}  → 해당 도구 호출 전에 자동으로 interrupt 발생
# checkpointer: interrupt 지점을 저장하는 체크포인터 (필수!)
checkpointer = InMemorySaver()  # 실제로는 PostgresSaver 등을 사용해요

hitl_agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    tools=[write_file, send_email],
    system_prompt=(
        "당신은 파일 관리와 이메일 발송을 도와주는 에이전트입니다. "
        "파일 저장이나 이메일 발송 전에 사용자의 확인을 받아야 해요."
    ),
    # interrupt_on은 딕셔너리 형태: {"도구명": True}
    interrupt_on={"write_file": True, "send_email": True},
    checkpointer=checkpointer,  # 상태 저장 필수!
)

# HITL 에이전트 생성 완료
# interrupt_on 도구: write_file, send_email
print(f"checkpointer: {type(checkpointer).__name__}")


checkpointer: InMemorySaver


In [10]:
# ---------------------------------------------------
# HITL 실행: interrupt 발생 → 인간 검토 → 재개
# ---------------------------------------------------
# thread_id: 어떤 대화 스레드인지 식별하는 고유 ID예요
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

# [1단계] 에이전트에게 파일 작성 요청
# --------------------------------------------------

# 파일 쓰기를 유도하는 요청이에요
inputs = {
    "messages": [HumanMessage(
        content="'report.txt'라는 파일에 'LangGraph 학습 완료' 라고 저장해줘"
    )]
}

# 에이전트 실행 → write_file 도구 호출 시 interrupt 발생
# Deep Agents의 stream 출력에서 messages 필드는 Overwrite 객체일 수 있어요
# Overwrite.value로 실제 메시지 리스트를 꺼내야 해요
interrupted = False
try:
    for chunk in hitl_agent.stream(inputs, config, stream_mode="updates"):
        for node_name, node_output in chunk.items():
            if not node_output or not isinstance(node_output, dict):
                continue
            if "messages" in node_output:
                msgs = node_output["messages"]
                # Overwrite 객체인 경우 .value로 실제 리스트를 꺼내요
                if hasattr(msgs, "value"):
                    msgs = msgs.value
                if not isinstance(msgs, list):
                    continue
                for msg in msgs:
                    if hasattr(msg, "tool_calls") and msg.tool_calls:
                        print(f"[{node_name}] 도구 호출 요청:")
                        for tc in msg.tool_calls:
                            print(f"  도구: {tc['name']}")
                            print(f"  인자: {tc['args']}")
                            #   → interrupt_on 대상 → 실행 중단!
                        interrupted = True
except Exception as e:
    # GraphInterrupt 등 예외가 발생하면 interrupt가 성공적으로 발생한 것이에요
    print(f"interrupt 발생: {type(e).__name__}")
    interrupted = True

if interrupted:
    print()
    # → 에이전트가 write_file 호출 전에 멈췄어요!
    # → 상태가 checkpointer에 저장되었어요. 다음 셀에서 승인하거나 거부할 수 있어요.
else:
    # (interrupt 없이 완료 — 에이전트가 다른 도구를 선택했을 수 있어요)
    pass


[model] 도구 호출 요청:
  도구: write_file
  인자: {'filename': '/report.txt', 'content': 'LangGraph 학습 완료'}



In [11]:
# ---------------------------------------------------
# HITL 재개: Command(resume=...)로 인간 결정 전달
# ---------------------------------------------------
# Command는 LangGraph의 상태 제어 객체예요
# resume 값으로 인간의 결정(승인/거부)을 에이전트에 전달해요
from langgraph.types import Command

# [2단계] 인간 검토 후 승인 결정
# --------------------------------------------------

# ---------------------------------------------------
# 승인하는 경우: resume에 실제로 실행할 내용 전달
# ---------------------------------------------------
# decisions: 각 도구 호출에 대한 결정 (approve/edit/reject)
approve_command = Command(
    resume={"decisions": [{"type": "approve"}]}  # 승인(approve) 결정을 전달해요
)

# 저장된 상태에서 재개해요 (같은 thread_id 사용)
for chunk in hitl_agent.stream(approve_command, config, stream_mode="updates"):
    for node_name, node_output in chunk.items():
        if not node_output or not isinstance(node_output, dict):
            continue
        if "messages" in node_output:
            msgs = node_output["messages"]
            # Overwrite 객체인 경우 .value로 실제 리스트를 꺼내요
            if hasattr(msgs, "value"):
                msgs = msgs.value
            if not isinstance(msgs, list):
                continue
            for msg in msgs:
                if hasattr(msg, "content") and msg.content:
                    print(f"[{node_name}]: {msg.content[:200]}")


[HumanInTheLoopMiddleware.after_model]: [{'arguments': '{"filename":"/report.txt","content":"LangGraph 학습 완료"}', 'call_id': 'call_qSHO36ZZnUauoK1v48ruyQSv', 'name': 'write_file', 'type': 'function_call', 'id': 'fc_04767bb4dd46dde0006a2f7e427750819a96bccea3c3d8c73e', 'status': 'completed'}]
[tools]: '/report.txt' 파일에 15자 저장 완료
[model]: [{'type': 'text', 'text': "'report.txt' 파일에 'LangGraph 학습 완료'를 저장했습니다.", 'annotations': [], 'id': 'msg_04767bb4dd46dde0006a2f7e43d9b0819abd015170363dd76e'}]


## 7. 커스텀 BackendProtocol 구현

기본 제공 백엔드로 요구사항을 충족하기 어려운 경우, `BackendProtocol`을 직접 구현해서 커스텀 백엔드를 만들 수 있어요.

### BackendProtocol 인터페이스

커스텀 백엔드는 다음 메서드를 구현해야 해요:

| 메서드 | 설명 |
|--------|------|
| `ls(path)` | 디렉토리 목록 반환 |
| `read(path)` | 파일 내용 읽기 |
| `write(path, content)` | 파일 저장 |
| `edit(path, old, new)` | 파일 특정 부분 수정 |
| `glob(pattern)` | 패턴 매칭으로 파일 목록 |
| `grep(pattern, path)` | 파일 내 패턴 검색 |

> 💡 **실무 팁**: 커스텀 백엔드를 구현하면 S3, GCS 같은 클라우드 스토리지나 데이터베이스도 에이전트의 파일시스템으로 사용할 수 있어요. 에이전트 코드는 변경 없이 백엔드만 교체하면 돼요.

In [12]:
# ---------------------------------------------------
# 커스텀 BackendProtocol 구현 예시
# ---------------------------------------------------
# 딕셔너리를 파일시스템처럼 사용하는 인메모리 백엔드예요
# 실제로는 S3, GCS, DB 등 외부 저장소와 연동할 수 있어요

from typing import Optional

class InMemoryBackend:
    """딕셔너리 기반 인메모리 파일시스템 백엔드예요.

    BackendProtocol 인터페이스를 구현한 커스텀 백엔드 예시예요.
    실제 환경에서는 S3, GCS, 데이터베이스 등과 연동해요.
    """

    def __init__(self) -> None:
        # 파일 내용을 딕셔너리로 저장해요: {경로: 내용}
        self._files: dict[str, str] = {}

    def ls(self, path: str = "/") -> list[str]:
        """지정 경로의 파일 목록을 반환해요."""
        # 경로로 시작하는 파일들을 필터링해요
        prefix = path.rstrip("/") + "/"
        return [
            k for k in self._files
            if k.startswith(prefix) or path == "/"
        ]

    def read(self, path: str) -> str:
        """파일 내용을 읽어요."""
        if path not in self._files:
            return f"오류: '{path}' 파일을 찾을 수 없어요"
        return self._files[path]

    def write(self, path: str, content: str) -> str:
        """파일을 저장해요."""
        self._files[path] = content
        return f"'{path}' 저장 완료 ({len(content)}자)"

    def edit(self, path: str, old_text: str, new_text: str) -> str:
        """파일의 특정 텍스트를 수정해요."""
        if path not in self._files:
            return f"오류: '{path}' 파일을 찾을 수 없어요"
        if old_text not in self._files[path]:
            return f"오류: 수정할 텍스트를 찾을 수 없어요"
        self._files[path] = self._files[path].replace(old_text, new_text, 1)
        return f"'{path}' 수정 완료"

    def glob(self, pattern: str) -> list[str]:
        """패턴에 매칭되는 파일 경로 목록을 반환해요."""
        import fnmatch
        return [k for k in self._files if fnmatch.fnmatch(k, pattern)]

    def grep(self, pattern: str, path: Optional[str] = None) -> list[str]:
        """파일에서 패턴을 검색해요."""
        results = []
        targets = {path: self._files[path]} if path else self._files
        for file_path, content in targets.items():
            for i, line in enumerate(content.splitlines(), 1):
                if pattern in line:
                    results.append(f"{file_path}:{i}: {line}")
        return results


# ---------------------------------------------------
# 커스텀 백엔드 테스트
# ---------------------------------------------------
backend = InMemoryBackend()

# 파일 쓰기
print(backend.write("/notes/langgraph.md", "# LangGraph\n- StateGraph\n- Node\n- Edge"))
print(backend.write("/notes/deepagents.md", "# Deep Agents\n- Planning\n- Filesystem"))

# 파일 목록 조회
print(f"\n파일 목록: {backend.ls('/')}")

# 파일 읽기
print(f"\n파일 내용:\n{backend.read('/notes/langgraph.md')}")

# 패턴 검색
matches = backend.grep("StateGraph")
print(f"\n'StateGraph' 검색 결과: {matches}")

# 커스텀 BackendProtocol 구현 완료!
# 이 패턴으로 S3, GCS, DB 등 어떤 저장소도 Deep Agents의 파일시스템으로 연결할 수 있어요.


'/notes/langgraph.md' 저장 완료 (38자)
'/notes/deepagents.md' 저장 완료 (37자)

파일 목록: ['/notes/langgraph.md', '/notes/deepagents.md']

파일 내용:
# LangGraph
- StateGraph
- Node
- Edge

'StateGraph' 검색 결과: ['/notes/langgraph.md:2: - StateGraph']


In [13]:
# ---------------------------------------------------
# 커스텀 백엔드를 에이전트에 연결하기
# ---------------------------------------------------
# 커스텀 백엔드를 에이전트에 전달해요
custom_backend_agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    tools=[],
    system_prompt="당신은 파일 관리 에이전트입니다. 파일 시스템을 활용해 작업을 수행하세요.",
    backend=InMemoryBackend(),  # 커스텀 백엔드 연결!
)
# 커스텀 백엔드 에이전트 생성 완료


## 8. 실습: HITL 에이전트 완성하기

지금까지 배운 내용을 종합해서 파일 쓰기 전 승인을 요청하는 에이전트를 완성해봐요.

In [ ]:
# ============================================================
# 실습 정답 예시: 다중 도구 HITL 에이전트 만들기
# 힌트:
#   1. write_file과 send_email 두 도구를 모두 interrupt_on에 추가해요
#   2. 에이전트에게 먼저 파일을 쓰고, 그 다음 이메일을 발송하도록 요청해요
#   3. 각 도구 호출마다 interrupt가 발생하는 것을 확인해요
#   4. Command(resume={"decisions": [{"type": "approve"}, {"type": "approve"}]})로 승인해봐요
# 예상 결과: 파일 쓰기 → 승인 → 이메일 발송 → 승인 → 완료
# ============================================================

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
import uuid

# --- 실습용 도구들 (이미 위에서 정의됨) ---
# write_file, send_email 도구 재사용

multi_hitl_checkpointer = InMemorySaver()

multi_hitl_agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    tools=[write_file, send_email],
    system_prompt=(
        "당신은 승인 기반 업무 비서입니다. "
        "사용자가 요청하면 먼저 write_file로 보고서를 저장하고, "
        "그 다음 send_email로 저장 사실을 알려주세요. "
        "두 도구는 모두 인간 승인 후에만 실행됩니다."
    ),
    interrupt_on={"write_file": True, "send_email": True},
    checkpointer=multi_hitl_checkpointer,
)

thread_id_todo = str(uuid.uuid4())
config_todo = {"configurable": {"thread_id": thread_id_todo}}

print("interrupt_on 설정: {'write_file': True, 'send_email': True}")
print(f"checkpointer: {type(multi_hitl_checkpointer).__name__}")
print("다음 단계: 에이전트를 실행하면 첫 번째 승인 지점에서 멈추고, Command(resume=...)로 재개합니다.")


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **Planning**: `write_todos` 도구가 자동 내장되어, 복잡한 작업을 단계별 할일 목록으로 관리해요.
- **Filesystem 백엔드**: `StateBackend`, `FilesystemBackend`, `StoreBackend`, `LocalShellBackend`, `CompositeBackend` 중 용도에 맞게 선택해요.
- **Permissions**: `FilesystemPermission`으로 경로·연산별 접근 규칙을 선언형으로 관리해요.
- **Subagents**: 딕셔너리(`SubAgent`) 또는 `CompiledSubAgent`로 전문 서브에이전트를 정의하고, 메인 에이전트가 `task` 도구로 작업을 위임해요.
- **Context Management**: 큰 도구 입출력은 파일시스템으로 오프로딩되고, 히스토리가 커지면 요약 미들웨어가 동작해요.
- **Code Execution**: `LocalShellBackend`의 `execute` 도구 또는 원격 샌드박스를 도구로 래핑할 수 있어요.
- **HITL 적용**: 06장 미들웨어 HITL에서 배운 승인 정책을 Deep Agents 하네스의 `interrupt_on` 설정으로 적용해요.
- **커스텀 BackendProtocol**: `ls`, `read`, `write`, `edit`, `glob`, `grep` 메서드를 구현하면 다양한 저장소를 Deep Agents 파일시스템으로 연결할 수 있어요.


## 9. 5대 원칙 × 6능력 매트릭스

### Harness 5대 원칙

Deep Agents SDK는 에이전트의 자율 행동을 안전하고 신뢰성 있게 통제하기 위해 5가지 설계 원칙을 제시합니다.

| 원칙 | 설명 |
|------|------|
| **Constrain** (제한) | 에이전트가 수행할 수 있는 행동의 범위를 명시적으로 좁힌다. 권한 규칙, 도구 세트 최소화, 샌드박스 격리가 대표적 수단이다. |
| **Inform** (정보 제공) | 에이전트에게 올바른 의사결정에 필요한 컨텍스트를 충분히 공급한다. 계획 상태, 파일 내용, 메모리, 시스템 프롬프트가 해당된다. |
| **Verify** (검증) | 에이전트가 수행한 작업의 결과물이 올바른지 사후 확인한다. 서브에이전트 검토, 코드 실행 결과 확인, HITL 승인이 그 예다. |
| **Correct** (교정) | 잘못된 결과를 탐지한 뒤 수정 경로를 실행한다. 서브에이전트의 비판적 피드백, 코드 재실행, 계획 갱신이 포함된다. |
| **HITL** (인간 개입) | 민감하거나 비용이 큰 작업에 대해 인간의 명시적 승인을 요구한다. `interrupt_on` 파라미터로 구성한다. |

---

### 능력 × 원칙 매핑 테이블

| 능력 \ 원칙 | Constrain | Inform | Verify | Correct | HITL |
|---|---|---|---|---|---|
| Planning (write_todos) | | ✓ | ✓ | | |
| Filesystem | ✓ | ✓ | | | |
| Subagents | ✓ | | ✓ | ✓ | |
| Context Engineering | | ✓ | | | |
| Code Execution | | | ✓ | ✓ | |
| HITL | ✓ | | ✓ | ✓ | ✓ |

---

### 각 매핑의 근거

**Planning (write_todos) — Inform, Verify**
- *Inform*: `write_todos`는 다음에 해야 할 작업과 현재 상태(pending/in_progress/completed)를 에이전트 자신과 사용자에게 명확히 제공한다.
- *Verify*: 계획 목록은 완료된 단계를 추적하여 누락된 작업이 없는지 사후 검증하는 역할을 한다.

**Filesystem — Constrain, Inform**
- *Constrain*: 권한 규칙(permissions)과 경로 기반 glob 패턴으로 에이전트가 접근할 수 있는 파일 범위를 제한한다.
- *Inform*: `read_file`, `ls`, `grep` 등으로 에이전트에게 작업 맥락(파일 내용, 디렉토리 구조)을 공급한다.

**Subagents — Constrain, Verify, Correct**
- *Constrain*: 각 서브에이전트는 최소한의 도구 세트와 격리된 컨텍스트를 가져 메인 에이전트를 보호한다.
- *Verify*: critic 서브에이전트처럼 독립된 관점에서 산출물을 검토할 수 있다.
- *Correct*: 검토 결과를 바탕으로 메인 에이전트가 산출물을 수정하거나 재시도하는 교정 루프를 구성한다.

**Context Engineering — Inform**
- *Inform*: 시스템 프롬프트, 메모리(AGENTS.md), 스킬 로딩, 자동 압축을 통해 에이전트가 장기 작업 내내 올바른 정보를 보유하도록 설계된 능력이다.

**Code Execution — Verify, Correct**
- *Verify*: 코드를 격리 샌드박스에서 실행하여 결과가 실제로 올바른지 확인한다(단순 생성이 아닌 실행 검증).
- *Correct*: 실행 오류 발생 시 에이전트가 코드를 수정하고 재실행하는 교정 루프를 자연스럽게 형성한다.

**HITL — Constrain, Verify, Correct, HITL**
- *Constrain*: `interrupt_on` 설정으로 특정 도구 실행 전 강제 중단점을 삽입한다.
- *Verify*: 인간이 행동 계획을 직접 확인하는 가장 강력한 검증 수단이다.
- *Correct*: 인간이 승인을 거부하거나 지시를 수정함으로써 에이전트 방향을 교정한다.
- *HITL*: 이 능력 자체가 원칙의 구현체다.


## 10. 통합 시나리오 — Planning + Filesystem + Subagents 조합

실제 과업에서는 6가지 능력이 동시에 동작합니다. 이 섹션에서는
"Planning → Filesystem 저장 → Subagent 검증 → 최종 합성" 흐름을
단일 에이전트로 구현합니다.

### 시나리오 개요

**질문**: "Python의 dataclass와 Pydantic BaseModel의 차이를 비교하고 추천 가이드를 작성해줘"

| 단계 | 수행 주체 | 설명 |
|------|----------|------|
| 1. 계획 수립 | 메인 에이전트 | `write_todos`로 비교 항목 분해 |
| 2. 비교 노트 작성 | 메인 에이전트 | `write_file`로 중간 결과 저장 |
| 3. 검토 요청 | critic 서브에이전트 | 파일을 읽고 누락 관점 지적 |
| 4. 최종 답변 합성 | 메인 에이전트 | 피드백 반영 후 최종 출력 |


### 환경 설정

In [15]:
from dotenv import load_dotenv
load_dotenv()


True

### 에이전트 구성

`critic` 서브에이전트는 메인 에이전트가 파일시스템에 저장한 비교 노트를
비판적으로 검토하고 누락된 관점을 지적합니다.

`create_deep_agent` 시그니처 (공식 문서 기준):

```python
create_deep_agent(
    model,
    tools,
    *,
    system_prompt,
    subagents,        # list[SubAgent | CompiledSubAgent]
    middleware,
    memory,
    backend,
    interrupt_on,
) -> CompiledStateGraph
```


In [16]:
from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI

# critic 서브에이전트 정의 (딕셔너리 형식)
critic_subagent = {
    "name": "critic",
    "description": (
        "메인 에이전트가 파일시스템에 작성한 비교 노트를 읽고 "
        "누락된 관점, 부정확한 내용, 보완이 필요한 부분을 지적합니다."
    ),
    "system_prompt": (
        "당신은 Python 전문가 리뷰어입니다. "
        "파일시스템에서 비교 노트를 읽은 뒤, 다음 기준으로 비판적 피드백을 제공하세요:\n"
        "1. 누락된 비교 항목(성능, 직렬화, 검증 동작, 상속, IDE 지원 등)\n"
        "2. 잘못 서술된 내용\n"
        "3. 실무 추천 가이드의 타당성\n"
        "피드백은 항목별 불릿으로 간결하게 작성하세요."
    ),
}

# 메인 에이전트 시스템 프롬프트
main_system_prompt = (
    "당신은 Python 기술 작가입니다. 다음 절차를 반드시 따르세요.\n\n"
    "1. write_todos 도구로 작업 계획을 먼저 수립합니다.\n"
    "   - todo 1: dataclass 특징 조사\n"
    "   - todo 2: Pydantic BaseModel 특징 조사\n"
    "   - todo 3: 비교 노트 파일 작성 (경로: /workspace/comparison.md)\n"
    "   - todo 4: critic 서브에이전트에 검토 요청\n"
    "   - todo 5: 피드백 반영 후 최종 답변 작성\n\n"
    "2. 각 todo를 완료한 뒤 상태를 completed로 업데이트합니다.\n\n"
    "3. critic 서브에이전트 검토 결과를 최종 답변에 반드시 반영합니다."
)

# 에이전트 생성
agent = create_deep_agent(
    model=ChatOpenAI(model="gpt-4o-mini"),
    tools=[],  # built-in 도구(write_todos, read_file, write_file 등)만 사용
    system_prompt=main_system_prompt,
    subagents=[critic_subagent],
)

# 에이전트 생성 완료
print(f"에이전트 타입: {type(agent).__name__}")


에이전트 타입: CompiledStateGraph


### 에이전트 실행

아래 셀을 실행하면 에이전트가 계획 수립 → 파일 저장 → critic 검토 → 최종 합성 순서로 동작합니다.
실제 API 호출이 발생하므로 `.env`에 `OPENAI_API_KEY`가 설정되어 있어야 합니다.


In [17]:
# 에이전트 실행
user_question = (
    "Python의 dataclass와 Pydantic의 BaseModel 차이를 비교하고 "
    "어떤 상황에서 무엇을 선택해야 하는지 추천 가이드를 작성해줘."
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": user_question}]}
)

# 최종 메시지 출력
final_message = result["messages"][-1]
# === 최종 답변 ===
print(final_message.content)


### Comparison of Python's `dataclass` and Pydantic's `BaseModel`

#### Dataclass Characteristics

1. **Automatic Method Generation**: Reduces boilerplate with automatic generation of methods like `__init__`, `__repr__`, `__eq__`, and more.

2. **Type Annotations**: Supports type checking and improves documentation through type-hinted attributes.

3. **Default Values**: Field default values for optional attributes.

4. **Immutability**: Supports frozen instances with `frozen=True`. Modifications post-initialization are restricted.

5. **Custom Metadata**: Use of `dataclasses.field()` for custom field behavior.

#### Pydantic BaseModel Characteristics

1. **Data Validation**: Provides powerful data validation using Python type hints to ensure data integrity and type checking.

2. **Nested Models**: Supports nested models for complex data structures, with automatic validation.

3. **Serialization/Deserialization**: Easily serializes and deserializes via JSON or other formats, with automa

### 이 시나리오에서 발동된 Harness 원칙 추적

에이전트 실행을 회고하면 각 단계가 5대 원칙 중 어느 것을 사용했는지 확인할 수 있습니다.

| 단계 | 사용 능력 | 발동된 원칙 | 구체적 동작 |
|------|----------|------------|------------|
| 계획 수립 | Planning (`write_todos`) | **Inform** | 5개 todo를 구조화해 에이전트 자신이 진행 상황을 파악 |
| 계획 수립 | Planning (`write_todos`) | **Verify** | todo 상태(pending → in_progress → completed)로 누락 작업 검증 |
| 비교 노트 저장 | Filesystem (`write_file`) | **Constrain** | `/workspace/` 경로로 파일 범위 한정 |
| 비교 노트 저장 | Filesystem (`write_file`) | **Inform** | 중간 결과를 파일에 보존하여 서브에이전트와 공유 |
| critic 검토 요청 | Subagents | **Constrain** | critic은 최소 권한(파일 읽기 + 피드백 작성)으로 격리 |
| critic 검토 결과 수신 | Subagents | **Verify** | 독립적 관점에서 비교 노트의 완결성 확인 |
| 피드백 반영 | Subagents | **Correct** | critic 지적 사항을 최종 답변에 반영하여 품질 교정 |

**관찰 포인트**

- Planning과 Filesystem은 Inform·Verify를 통해 에이전트가 "무엇을 알고 있는가"를 강화합니다.
- Subagents는 Constrain·Verify·Correct를 통해 "결과물이 충분히 좋은가"를 보장합니다.
- 이 시나리오에서 HITL은 사용하지 않았지만, `interrupt_on={"write_file": True}` 설정으로
  파일 저장 전 인간 승인을 추가할 수 있습니다.
- Context Engineering은 `system_prompt`와 자동 압축을 통해 전체 흐름에 묵시적으로 관여합니다.

---

**참고 문서**
- [Harness capabilities](https://docs.langchain.com/oss/python/deepagents/harness.md)
- [Subagents](https://docs.langchain.com/oss/python/deepagents/subagents.md)
- [Customize Deep Agents](https://docs.langchain.com/oss/python/deepagents/customization.md)


## 다음 노트북 예고

다음 `03-Context-Engineering.ipynb`에서는 **Deep Agents의 5가지 컨텍스트 타입**을 배워요. 입력(Input), 런타임(Runtime), 압축(Compaction), 격리(Isolation), 메모리(Memory) 각각이 언제 작동하고 어떻게 설계하는지, 시스템 프롬프트는 어디까지 정교하게 써야 하는지 다뤄요. 이번에 만나본 하네스 능력이 "왜 그렇게 설계됐는지" 이유가 드러나요.